In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

### Test Data Loader

In [2]:
from src.data_manager.data_loader import HeteroDataLoader

loader = HeteroDataLoader("../configs/base.yaml")

print("Loading Training Samples...")
train_data_list = loader.load_all_samples("train")

if train_data_list:
    sample = train_data_list[0]
    print(f"\nLoaded {len(train_data_list)} samples.")
    print(f"Sample {sample['sample_name']} breakdown:")
    print(f"   - Beams: {len(sample['nodes']['beam'])}")
    print(f"   - Columns: {len(sample['nodes']['column'])}")
    print(f"   - Total Edges: {len(sample['edges_raw'])}")

14:24:22 | 📁 DATA   | ℹ️  INFO     | Initializing HeteroDataLoader with config: ../configs/base.yaml
14:24:22 | 📁 DATA   | ℹ️  INFO     | DataLoader initialized successfully
Loading Training Samples...
14:24:22 | 📁 DATA   | ℹ️  INFO     | Loading all samples from split: train
14:24:22 | 📁 DATA   | ℹ️  INFO     | Found 254 sample folders
14:24:22 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_1 (train)
14:24:22 | 📁 DATA   | ❌ ERROR    | Sample sample_1: Missing files: Feature file
14:24:22 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_10 (train)
14:24:22 | 📁 DATA   | ❌ ERROR    | Sample sample_10: Missing files: Feature file
14:24:22 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_100 (train)
14:24:22 | 📁 DATA   | ❌ ERROR    | Sample sample_100: Missing files: Feature file
14:24:22 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_101 (train)
14:24:23 | 📁 DATA   | ❌ ERROR    | Sample sample_101: Missing files: Feature file
14:24:23 | 📁 DATA   | ℹ️  INFO     | Loading sample

In [3]:
train_data_list[0]

IndexError: list index out of range

In [ ]:
"""
Diagnostic Script for Edge-Node Matching Issues
"""
import pandas as pd
import numpy as np
from src.data_manager.data_loader import HeteroDataLoader

def diagnose_sample(sample_name="sample_1"):
    """Diagnose a single sample to understand node isolation"""
    
    loader = HeteroDataLoader("../configs/base.yaml")
    sample = loader.load_sample(sample_name, split="train")
    
    if not sample:
        print(f"❌ Could not load sample {sample_name}")
        return
    
    print(f"\n🔍 DIAGNOSTICS FOR SAMPLE: {sample_name}")
    print("="*60)
    
    # Get data
    beam_df = sample["nodes"]["beam"]
    column_df = sample["nodes"]["column"]
    edges_df = sample["edges_raw"]
    
    print(f"\n📊 NODE COUNTS:")
    print(f"  Beams: {len(beam_df)} rows")
    print(f"  Columns: {len(column_df)} rows")
    print(f"  Total nodes: {len(beam_df) + len(column_df)}")
    
    print(f"\n📊 EDGE COUNTS:")
    print(f"  Total edges in file: {len(edges_df)}")
    
    # Get all row indices present in nodes
    all_row_indices = set()
    for df in [beam_df, column_df]:
        if 'row_index' in df.columns:
            all_row_indices.update(df['row_index'].astype(int).tolist())
    
    print(f"\n📊 ROW INDEX ANALYSIS:")
    print(f"  Unique row indices in nodes: {len(all_row_indices)}")
    if all_row_indices:
        print(f"  Range: [{min(all_row_indices)}, {max(all_row_indices)}]")
    
    # Check edge indices
    if 'Source' in edges_df.columns and 'Target' in edges_df.columns:
        source_indices = set(edges_df['Source'].dropna().astype(int).tolist())
        target_indices = set(edges_df['Target'].dropna().astype(int).tolist())
        all_edge_indices = source_indices.union(target_indices)
        
        print(f"\n📊 EDGE INDEX ANALYSIS:")
        print(f"  Unique source indices: {len(source_indices)}")
        print(f"  Unique target indices: {len(target_indices)}")
        print(f"  Total unique edge indices: {len(all_edge_indices)}")
        
        if all_edge_indices:
            print(f"  Range: [{min(all_edge_indices)}, {max(all_edge_indices)}]")
        
        # Find mismatches
        missing_in_nodes = all_edge_indices - all_row_indices
        missing_in_edges = all_row_indices - all_edge_indices
        
        print(f"\n⚠️  MISMATCH ANALYSIS:")
        print(f"  Edge indices missing from nodes: {len(missing_in_nodes)}")
        if missing_in_nodes:
            print(f"    Example missing: {sorted(list(missing_in_nodes))[:10]}")
        
        print(f"  Node indices missing from edges: {len(missing_in_edges)}")
        if missing_in_edges:
            print(f"    Example missing: {sorted(list(missing_in_edges))[:10]}")
        
        # Calculate isolation
        if all_row_indices:
            connected_via_edges = all_row_indices.intersection(all_edge_indices)
            isolated = all_row_indices - connected_via_edges
            
            print(f"\n🔗 CONNECTIVITY ANALYSIS:")
            print(f"  Connected nodes: {len(connected_via_edges)} ({len(connected_via_edges)/len(all_row_indices)*100:.1f}%)")
            print(f"  Isolated nodes: {len(isolated)} ({len(isolated)/len(all_row_indices)*100:.1f}%)")
            
            # Check by node type
            beam_indices = set(beam_df['row_index'].astype(int).tolist()) if 'row_index' in beam_df.columns else set()
            column_indices = set(column_df['row_index'].astype(int).tolist()) if 'row_index' in column_df.columns else set()
            
            print(f"\n📈 BY NODE TYPE:")
            print(f"  Beam isolated: {len(beam_indices - connected_via_edges)}/{len(beam_indices)}")
            print(f"  Column isolated: {len(column_indices - connected_via_edges)}/{len(column_indices)}")
    
    # Check first few rows of each file
    print(f"\n📄 SAMPLE DATA VIEW:")
    print("\nBeams DataFrame (first 3 rows):")
    print(beam_df[['row_index', 'Ele_Type']].head(3).to_string())
    
    print("\nColumns DataFrame (first 3 rows):")
    print(column_df[['row_index', 'Ele_Type']].head(3).to_string())
    
    print("\nEdges DataFrame (first 5 rows):")
    if not edges_df.empty:
        print(edges_df[['Source', 'Target']].head(5).to_string())
    
    return sample

# Run diagnostics
if __name__ == "__main__":
    # Test with first few samples
    for sample_name in ["sample_1", "sample_10", "sample_100"]:
        diagnose_sample(sample_name)
        print("\n" + "="*60 + "\n")

19:49:40 | 📁 DATA   | ℹ️  INFO     | Initializing HeteroDataLoader with config: ../configs/base.yaml


19:49:40 | 📁 DATA   | ℹ️  INFO     | DataLoader initialized successfully
19:49:40 | 📁 DATA   | ℹ️  INFO     | Loading sample: sample_1 (train)
19:49:41 | 📁 DATA   | ℹ️  INFO     | Successfully loaded sample sample_1

🔍 DIAGNOSTICS FOR SAMPLE: sample_1

📊 NODE COUNTS:
  Beams: 371 rows
  Columns: 103 rows
  Total nodes: 474

📊 EDGE COUNTS:
  Total edges in file: 2804

📊 ROW INDEX ANALYSIS:
  Unique row indices in nodes: 474
  Range: [0, 473]

📊 EDGE INDEX ANALYSIS:
  Unique source indices: 458
  Unique target indices: 458
  Total unique edge indices: 458
  Range: [0, 473]

⚠️  MISMATCH ANALYSIS:
  Edge indices missing from nodes: 0
  Node indices missing from edges: 16
    Example missing: [176, 177, 178, 179, 180, 181, 182, 183, 192, 193]

🔗 CONNECTIVITY ANALYSIS:
  Connected nodes: 458 (96.6%)
  Isolated nodes: 16 (3.4%)

📈 BY NODE TYPE:
  Beam isolated: 16/371
  Column isolated: 0/103

📄 SAMPLE DATA VIEW:

Beams DataFrame (first 3 rows):
    row_index  Ele_Type
87         87         